# Nifty 50 Signal — Training v5 (Direction × Big-Move stack)

**v4 outcome:** 36% directional precision on holdout. Loses to mean-reversion baseline (52%).

**v5 thesis:** Predicting direction every day is too hard. Predicting *when to trade at all* is easier.

**v5 architecture:**
1. **Big-move binary model**: predicts whether tomorrow's absolute move will be in the top 30% of historical moves (≈ 'tradeable day' filter)
2. **Directional model**: same 3-class as v4 (BUY_CALL / NO_TRADE / BUY_PUT)
3. **Stacked gate**: trade ONLY when `big_move_prob ≥ 0.50` AND `direction_conf ≥ 0.50`

**Expected:** Big-move model hits 60-65% precision (easier problem). Stacked trades fire ~5-10% of days at 55-60% precision. Low frequency, real edge.

**Setup:** Runtime → T4 GPU. CSVs in `/MyDrive/nifty_signal/data/`.

In [ ]:
!pip install -q ta xgboost lightgbm catboost scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
DATA_DIR = '/content/drive/MyDrive/nifty_signal/data'
MODELS_DIR = '/content/drive/MyDrive/nifty_signal/models'
os.makedirs(MODELS_DIR, exist_ok=True)

files = os.listdir(DATA_DIR)
def find(patterns):
    for p in patterns:
        for f in files:
            if re.search(p, f, re.IGNORECASE):
                return f
    return None

NIFTY_DAILY  = find([r'nifty.*50.*daily', r'^nifty50_daily', r'^nifty.*daily'])
NIFTY_HOURLY = find([r'nifty.*50.*hour', r'^nifty50_hour'])
VIX_FILE     = find([r'india.*vix', r'^vix', r'_vix\.csv'])
BNF_FILE     = find([r'bank.*nifty', r'nifty.*bank'])

print(f'Nifty daily:  {NIFTY_DAILY}')
print(f'Nifty hourly: {NIFTY_HOURLY}')
print(f'India VIX:    {VIX_FILE}')
print(f'Bank Nifty:   {BNF_FILE}')

In [ ]:
import pandas as pd, numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    for c in ('date','timestamp','time'):
        if c in df.columns and 'datetime' not in df.columns:
            df = df.rename(columns={c:'datetime'}); break
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.tz_localize(None).dt.normalize() if df['datetime'].dt.tz else df['datetime'].dt.normalize()
    return df.sort_values('datetime').reset_index(drop=True)

def add_nifty_features(df):
    out = df.copy()
    c, h, l, v = out['close'], out['high'], out['low'], out['volume']
    out['rsi_14'] = RSIIndicator(c, 14).rsi()
    out['macd_hist'] = MACD(c, 26, 12, 9).macd_diff()
    ema9, ema21, ema50 = EMAIndicator(c, 9).ema_indicator(), EMAIndicator(c, 21).ema_indicator(), EMAIndicator(c, 50).ema_indicator()
    out['ema_cross_9_21']  = (ema9 > ema21).astype(int)
    out['ema_cross_21_50'] = (ema21 > ema50).astype(int)
    out['atr_14'] = AverageTrueRange(h, l, c, 14).average_true_range()
    bb = BollingerBands(c, 20, 2)
    out['bb_pct_b'] = (c - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband())
    out['adx_14'] = ADXIndicator(h, l, c, 14).adx()
    for lag in (1,3,5,10,20): out[f'ret_{lag}d'] = c.pct_change(lag)
    rets = c.pct_change()
    for w in (5,10,20):
        out[f'vol_{w}d']  = rets.rolling(w).std()
        out[f'mean_{w}d'] = rets.rolling(w).mean()
        out[f'high_{w}d_dist'] = (c - h.rolling(w).max()) / h.rolling(w).max()
        out[f'low_{w}d_dist']  = (c - l.rolling(w).min()) / l.rolling(w).min()
    atr_50 = AverageTrueRange(h, l, c, 50).average_true_range()
    out['vol_regime'] = out['atr_14'] / atr_50
    out['atr_pct']    = out['atr_14'] / c
    v_safe = v.replace(0, np.nan)
    out['log_vol']     = np.log1p(v_safe).fillna(0)
    vma20, vstd20 = v_safe.rolling(20).mean(), v_safe.rolling(20).std()
    out['vol_vs_ma20'] = (v_safe / vma20).fillna(1.0)
    out['vol_zscore']  = ((v_safe - vma20) / vstd20).fillna(0.0)
    out['gap_pct'] = (out['open'] - c.shift(1)) / c.shift(1)
    out['rsi_divergence'] = (np.sign(c.pct_change(14)) * np.sign(out['rsi_14'].diff(14))).fillna(0)
    dt = pd.to_datetime(out['datetime'])
    out['day_of_week'] = dt.dt.dayofweek
    out['dte'] = (3 - dt.dt.dayofweek) % 7
    return out

FEATURES_NIFTY = [
    'rsi_14','macd_hist','ema_cross_9_21','ema_cross_21_50','atr_14','bb_pct_b','adx_14',
    'ret_1d','ret_3d','ret_5d','ret_10d','ret_20d',
    'vol_5d','vol_10d','vol_20d','mean_5d','mean_10d','mean_20d',
    'high_5d_dist','high_10d_dist','high_20d_dist','low_5d_dist','low_10d_dist','low_20d_dist',
    'vol_regime','atr_pct','log_vol','vol_vs_ma20','vol_zscore','gap_pct','rsi_divergence',
    'day_of_week','dte',
]

In [ ]:
def hourly_to_daily(hourly_df):
    h = hourly_df.copy()
    h['intraday_ret'] = (h['close'] - h['open']) / h['open']
    g = h.groupby('date').agg(
        intraday_total_return=('close', lambda s: (s.iloc[-1]-s.iloc[0])/s.iloc[0] if len(s)>1 else 0),
        intraday_max_drawdown=('close', lambda s: (s.min()-s.iloc[0])/s.iloc[0] if len(s)>1 else 0),
        intraday_vol=('intraday_ret','std'),
        intraday_volume_total=('volume','sum'),
        intraday_n_bars=('close','count'),
    ).reset_index()
    g['intraday_volume_total'] = np.log1p(g['intraday_volume_total'].fillna(0))
    g['intraday_vol'] = g['intraday_vol'].fillna(0)
    return g

def build_vix_features(vix_df):
    v = vix_df.copy()
    vix_col = 'close' if 'close' in v.columns else ('value' if 'value' in v.columns else None)
    if vix_col is None: raise ValueError(f'VIX needs close/value column')
    v = v[['date', vix_col]].rename(columns={vix_col:'vix'})
    v['vix_change_1d'] = v['vix'].pct_change(1)
    v['vix_change_5d'] = v['vix'].pct_change(5)
    v['vix_vs_ma20']   = v['vix'] / v['vix'].rolling(20).mean()
    v['vix_high'] = (v['vix'] > 20).astype(int)
    v['vix_low']  = (v['vix'] < 14).astype(int)
    return v

def build_bnf_features(bnf_df, nifty_close):
    b = bnf_df.copy()
    b = b[['date','close']].rename(columns={'close':'bnf_close'})
    b['bnf_ret_1d'] = b['bnf_close'].pct_change(1)
    b['bnf_ret_5d'] = b['bnf_close'].pct_change(5)
    b['bnf_ret_20d']= b['bnf_close'].pct_change(20)
    bnf_rets = b['bnf_close'].pct_change()
    nrets = nifty_close.pct_change().iloc[-len(bnf_rets):].reset_index(drop=True)
    b['bnf_nifty_corr_20'] = bnf_rets.rolling(20).corr(nrets)
    b['bnf_nifty_divergence'] = np.sign(b['bnf_ret_1d']) - np.sign(nrets)
    return b[['date','bnf_ret_1d','bnf_ret_5d','bnf_ret_20d','bnf_nifty_corr_20','bnf_nifty_divergence']]

FEATURES_INTRADAY = ['intraday_total_return','intraday_max_drawdown','intraday_vol','intraday_volume_total','intraday_n_bars']
FEATURES_VIX = ['vix','vix_change_1d','vix_change_5d','vix_vs_ma20','vix_high','vix_low']
FEATURES_BNF = ['bnf_ret_1d','bnf_ret_5d','bnf_ret_20d','bnf_nifty_corr_20','bnf_nifty_divergence']
ALL_FEATURES = FEATURES_NIFTY + FEATURES_INTRADAY + FEATURES_VIX + FEATURES_BNF
print(f'{len(ALL_FEATURES)} total features')

In [ ]:
# Load + merge everything
nifty = load_csv(f'{DATA_DIR}/{NIFTY_DAILY}')
feats = add_nifty_features(nifty)
n_years = (feats['date'].max() - feats['date'].min()).days / 365.25
print(f'Loaded Nifty: {len(feats)} rows over {n_years:.1f} years '
      f'({feats["date"].min().date()} → {feats["date"].max().date()})')

if NIFTY_HOURLY:
    hourly = load_csv(f'{DATA_DIR}/{NIFTY_HOURLY}')
    intra = hourly_to_daily(hourly)
    feats = feats.merge(intra, on='date', how='left')
    for c in FEATURES_INTRADAY: feats[c] = feats[c].fillna(0)
else:
    for c in FEATURES_INTRADAY: feats[c] = 0

if VIX_FILE:
    vix = build_vix_features(load_csv(f'{DATA_DIR}/{VIX_FILE}'))
    feats = feats.merge(vix, on='date', how='left')
    for c in FEATURES_VIX: feats[c] = feats[c].ffill().bfill()
else:
    for c in FEATURES_VIX: feats[c] = 0

if BNF_FILE:
    bnf = build_bnf_features(load_csv(f'{DATA_DIR}/{BNF_FILE}'), nifty.set_index('date')['close'])
    feats = feats.merge(bnf, on='date', how='left')
    for c in FEATURES_BNF: feats[c] = feats[c].ffill().fillna(0)
else:
    for c in FEATURES_BNF: feats[c] = 0

# Add targets
feats['next_return'] = feats['close'].pct_change().shift(-1)
feats['next_abs_return'] = feats['next_return'].abs()

# Sanitize
feats[ALL_FEATURES] = feats[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)
feats = feats.dropna(subset=['next_return']).reset_index(drop=True)
feats = feats.dropna(subset=ALL_FEATURES).reset_index(drop=True)

# Clip outliers
for c in ALL_FEATURES:
    if feats[c].dtype.kind in 'fc':
        q01, q99 = feats[c].quantile([0.001, 0.999])
        feats[c] = feats[c].clip(q01, q99)

print(f'\nFinal dataset: {len(feats)} rows, {feats["date"].min().date()} → {feats["date"].max().date()}')

## Define the two label sets

**Direction**: 3-class quantile (bottom-33 = BUY_PUT, mid-34 = NO_TRADE, top-33 = BUY_CALL).

**Big-move binary**: 1 if `|next_return|` is in top 30% of training fold's absolute returns, else 0. This is **the day the model decides 'is tomorrow worth trading?'**.

In [ ]:
def make_dir_labels(next_returns, q_low=0.33, q_high=0.67):
    lo, hi = next_returns.quantile(q_low), next_returns.quantile(q_high)
    lab = pd.Series(1, index=next_returns.index)
    lab[next_returns > hi] = 2
    lab[next_returns < lo] = 0
    return lab.values, lo, hi

def make_bigmove_labels(next_abs_returns, q_threshold=0.70):
    """1 if |return| in top (1-q_threshold) = top 30% by default."""
    cutoff = next_abs_returns.quantile(q_threshold)
    return (next_abs_returns > cutoff).astype(int).values, cutoff

# Holdout split
HOLDOUT_DAYS = 90
cutoff_date = feats['date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
df_train = feats[feats['date'] <= cutoff_date].reset_index(drop=True)
df_holdout = feats[feats['date'] > cutoff_date].reset_index(drop=True)
print(f'Train pool: {len(df_train)} rows  ({df_train["date"].min().date()} → {df_train["date"].max().date()})')
print(f'Holdout:    {len(df_holdout)} rows  ({df_holdout["date"].min().date()} → {df_holdout["date"].max().date()})')

X_pool   = df_train[ALL_FEATURES].values
ret_pool = df_train['next_return'].values
abs_pool = df_train['next_abs_return'].values

## Walk-forward CV — direction & big-move models

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

def xgb_direction():
    return XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.04,
                         objective='multi:softprob', num_class=3,
                         eval_metric='mlogloss', tree_method='hist', device='cuda',
                         reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8)

def xgb_bigmove():
    return XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.04,
                         objective='binary:logistic',
                         eval_metric='logloss', tree_method='hist', device='cuda',
                         reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8)

tscv = TimeSeriesSplit(n_splits=5)
dir_scores, big_scores, stacked_scores = [], [], []

for fold, (tr, te) in enumerate(tscv.split(X_pool), 1):
    # Direction labels (computed on train fold only)
    train_ret = pd.Series(ret_pool[tr])
    _, lo, hi = make_dir_labels(train_ret)
    full = pd.Series(ret_pool)
    y_dir = pd.Series(1, index=full.index); y_dir[full > hi] = 2; y_dir[full < lo] = 0
    y_dir = y_dir.values
    y_dir_tr, y_dir_te = y_dir[tr], y_dir[te]
    
    # Big-move labels (cutoff from train fold)
    train_abs = pd.Series(abs_pool[tr])
    _, cutoff_abs = make_bigmove_labels(train_abs)
    y_big = (pd.Series(abs_pool) > cutoff_abs).astype(int).values
    y_big_tr, y_big_te = y_big[tr], y_big[te]
    
    X_tr, X_te = X_pool[tr], X_pool[te]
    sw_dir = compute_sample_weight('balanced', y_dir_tr)
    sw_big = compute_sample_weight('balanced', y_big_tr)
    
    # Train direction
    md = xgb_direction(); md.fit(X_tr, y_dir_tr, sample_weight=sw_dir)
    dir_proba = md.predict_proba(X_te); dir_pred = dir_proba.argmax(axis=1); dir_conf = dir_proba.max(axis=1)
    dir_mask = dir_pred != 1
    dir_prec = (dir_pred[dir_mask] == y_dir_te[dir_mask]).mean() if dir_mask.any() else 0
    
    # Train big-move
    mb = xgb_bigmove(); mb.fit(X_tr, y_big_tr, sample_weight=sw_big)
    big_proba = mb.predict_proba(X_te)[:, 1]; big_pred = (big_proba >= 0.5).astype(int)
    big_prec = precision_score(y_big_te, big_pred, zero_division=0)
    big_recall = recall_score(y_big_te, big_pred, zero_division=0)
    
    # Stacked: direction precision when both gates fire (big >= 0.5 AND dir_conf >= 0.5)
    stacked = (big_proba >= 0.5) & (dir_mask) & (dir_conf >= 0.5)
    if stacked.any():
        stacked_prec = (dir_pred[stacked] == y_dir_te[stacked]).mean()
    else:
        stacked_prec = float('nan')
    
    dir_scores.append(dir_prec)
    big_scores.append(big_prec)
    stacked_scores.append((stacked_prec, int(stacked.sum())))
    print(f'Fold {fold}: dir_prec={dir_prec:.3f}  big_prec={big_prec:.3f} (recall {big_recall:.2f})  '
          f'stacked_prec={stacked_prec:.3f} (n={int(stacked.sum())})')

print(f'\n=== Mean across folds ===')
print(f'Direction precision:    {np.mean(dir_scores):.3f}')
print(f'Big-move precision:     {np.mean(big_scores):.3f}')
valid_stacked = [s for s, n in stacked_scores if n > 0]
total_stacked = sum(n for _, n in stacked_scores)
print(f'STACKED gated precision: {np.mean(valid_stacked):.3f}  (total {total_stacked} stacked trades over 5 folds)')

## Final train + holdout evaluation

In [ ]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import joblib, json

# Direction labels on full train pool
_, lo, hi = make_dir_labels(pd.Series(ret_pool))
y_dir_full = pd.Series(1, index=range(len(ret_pool)))
y_dir_full[pd.Series(ret_pool) > hi] = 2
y_dir_full[pd.Series(ret_pool) < lo] = 0
y_dir_full = y_dir_full.values

_, cutoff_abs = make_bigmove_labels(pd.Series(abs_pool))
y_big_full = (pd.Series(abs_pool) > cutoff_abs).astype(int).values

sw_dir = compute_sample_weight('balanced', y_dir_full)
sw_big = compute_sample_weight('balanced', y_big_full)

# --- Direction ensemble ---
xgb_dir = xgb_direction(); xgb_dir.fit(X_pool, y_dir_full, sample_weight=sw_dir)
lgbm_dir = LGBMClassifier(n_estimators=400, learning_rate=0.04, num_class=3,
                          objective='multiclass', class_weight='balanced', verbose=-1)
lgbm_dir.fit(X_pool, y_dir_full)
cat_dir = CatBoostClassifier(iterations=400, depth=5, learning_rate=0.04,
                              loss_function='MultiClass', auto_class_weights='Balanced',
                              task_type='GPU', verbose=0)
cat_dir.fit(X_pool, y_dir_full)

# Meta on held-out 20%
split = int(len(X_pool) * 0.8)
Xm_tr, Xm_val = X_pool[:split], X_pool[split:]
ym_tr, ym_val = y_dir_full[:split], y_dir_full[split:]
swm = compute_sample_weight('balanced', ym_tr)
xgb_m = xgb_direction(); xgb_m.fit(Xm_tr, ym_tr, sample_weight=swm)
lgbm_m = LGBMClassifier(n_estimators=400, num_class=3, objective='multiclass', class_weight='balanced', verbose=-1); lgbm_m.fit(Xm_tr, ym_tr)
cat_m  = CatBoostClassifier(iterations=400, loss_function='MultiClass', auto_class_weights='Balanced', task_type='GPU', verbose=0); cat_m.fit(Xm_tr, ym_tr)
base_val = np.hstack([xgb_m.predict_proba(Xm_val), lgbm_m.predict_proba(Xm_val), cat_m.predict_proba(Xm_val)])
meta = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.3); meta.fit(base_val, ym_val)
cal_meta = CalibratedClassifierCV(meta, method='sigmoid', cv='prefit'); cal_meta.fit(base_val, ym_val)

# --- Big-move model (single XGB is fine for binary) ---
xgb_big = xgb_bigmove(); xgb_big.fit(X_pool, y_big_full, sample_weight=sw_big)

# --- Holdout eval ---
X_hold = df_holdout[ALL_FEATURES].values
ret_hold = df_holdout['next_return']
abs_hold = df_holdout['next_abs_return']
y_dir_hold = pd.Series(1, index=range(len(ret_hold)))
y_dir_hold[ret_hold.values > hi] = 2; y_dir_hold[ret_hold.values < lo] = 0
y_dir_hold = y_dir_hold.values
y_big_hold = (abs_hold.values > cutoff_abs).astype(int)

base_h = np.hstack([xgb_dir.predict_proba(X_hold), lgbm_dir.predict_proba(X_hold), cat_dir.predict_proba(X_hold)])
dir_pred_h = cal_meta.predict(base_h)
dir_proba_h = cal_meta.predict_proba(base_h)
dir_conf_h = dir_proba_h.max(axis=1)
big_proba_h = xgb_big.predict_proba(X_hold)[:, 1]

print('=== v5 HOLDOUT ===\n')

# Direction-only
dir_mask_h = dir_pred_h != 1
dir_prec_h = (dir_pred_h[dir_mask_h] == y_dir_hold[dir_mask_h]).mean() if dir_mask_h.any() else 0
print(f'Direction-only (baseline):')
print(f'  precision: {dir_prec_h:.3f}  ({dir_mask_h.sum()} directional preds)')

# Big-move alone
big_pred_h = (big_proba_h >= 0.5).astype(int)
big_prec_h = precision_score(y_big_hold, big_pred_h, zero_division=0)
big_rec_h = recall_score(y_big_hold, big_pred_h, zero_division=0)
print(f'\nBig-move binary:')
print(f'  precision: {big_prec_h:.3f}  recall: {big_rec_h:.3f}  ({big_pred_h.sum()} predicted big days)')
print(f'  Baseline (always YES): {y_big_hold.mean():.3f} precision')

# Stacked at multiple thresholds
print(f'\n=== STACKED GATE (big_prob >= big_thresh) AND (dir_conf >= dir_thresh) ===')
print(f'{"big":>5} {"dir":>5} {"n_trades":>9} {"precision":>10}  {"baseline_dir@conf":>20}')
for big_t in (0.40, 0.50, 0.60):
    for dir_t in (0.40, 0.50, 0.60):
        stacked = (big_proba_h >= big_t) & dir_mask_h & (dir_conf_h >= dir_t)
        if stacked.sum() == 0:
            print(f'{big_t:>5.2f} {dir_t:>5.2f} {0:>9d}  {"--":>10}  --')
            continue
        prec = (dir_pred_h[stacked] == y_dir_hold[stacked]).mean()
        # baseline: direction precision at the same dir_conf threshold, no big_move gate
        baseline = dir_mask_h & (dir_conf_h >= dir_t)
        baseline_prec = (dir_pred_h[baseline] == y_dir_hold[baseline]).mean() if baseline.any() else 0
        delta = prec - baseline_prec
        print(f'{big_t:>5.2f} {dir_t:>5.2f} {int(stacked.sum()):>9d}  {prec:>10.3f}  '
              f'{baseline_prec:.3f} (Δ {delta:+.3f})')

In [ ]:
# Save everything
joblib.dump(xgb_dir,  f'{MODELS_DIR}/xgboost.pkl')
joblib.dump(lgbm_dir, f'{MODELS_DIR}/lightgbm.pkl')
joblib.dump(cat_dir,  f'{MODELS_DIR}/catboost.pkl')
joblib.dump(cal_meta, f'{MODELS_DIR}/meta_rf.pkl')
joblib.dump(xgb_big,  f'{MODELS_DIR}/bigmove.pkl')  # NEW

with open(f'{MODELS_DIR}/label_thresholds.json', 'w') as f:
    json.dump({
        'q_low_return': float(lo), 'q_high_return': float(hi),
        'bigmove_abs_return_cutoff': float(cutoff_abs),
        'features': ALL_FEATURES,
        'all_features_considered': ALL_FEATURES,
        'training_years': round(n_years, 1),
        'training_rows': len(df_train),
        'holdout_dir_prec': float(dir_prec_h),
        'holdout_bigmove_prec': float(big_prec_h),
    }, f, indent=2)

print(f'\nSaved to {MODELS_DIR}:')
for f in sorted(os.listdir(MODELS_DIR)):
    sz = os.path.getsize(f'{MODELS_DIR}/{f}') / 1e6
    print(f'  {f}  ({sz:.2f} MB)')

## Decision guide

Look at the **stacked precision** table above:

- **Cell where (big ≥ 0.50, dir ≥ 0.50) has 10+ trades AND precision ≥ 0.55 AND Δ ≥ +0.05** → big win, ship v5. Backend will need a small update to use the big-move gate.
- **Stacked precision similar to baseline (Δ near 0)** → big-move filter isn't adding value. The model can't tell big from small days well. Stick with v4.
- **Big-move precision (alone) below baseline `y_big_hold.mean()`** → the binary target also isn't predictable. The data has very weak signal. Consider Path 3 alternatives (volatility, regime, intraday models).

If shipping, I'll patch `backend/signal_engine.py` to load both models and apply the stacked gate. Tell me the holdout table and I'll write the integration.